# Experiment 1: Spectral Saturation sweeps across datasets

This notebook reproduces the main experiments from the paper. We run training sample size sweeps ($K$) on several classification datasets (MNIST, Fashion-MNIST, Kuzushiji-MNIST, USPS, Breast Cancer, and CIFAR-10) using Logistic Regression.

For each dataset, we track:
1. **Classification Accuracy** on a held-out test set.
2. **Effective Rank** (erank) of the pooled within-class covariance matrix.
3. **Saturation ratio** $S(K) = \text{erank} / K$.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sys

# Add src to python path to allow importing modules
sys.path.append(os.path.abspath('..'))

from src.datasets import load_all_datasets, load_cifar10
from src.experiment import run_experiment, save_results, load_results
from src.visualization import plot_all_sweeps_grid

## 1. Load Data

We load pre-downloaded dataset npz files from `data/` and download CIFAR-10 if not available.

In [ ]:
datasets = load_all_datasets(data_dir='../data')
X_mnist, y_mnist = datasets['MNIST']
X_fashion, y_fashion = datasets['Fashion']
X_kuzushiji, y_kuzushiji = datasets['Kuzushiji']
X_usps, y_usps = datasets['USPS']
X_bc, y_bc = datasets['BreastCancer']

X_cifar, y_cifar = load_cifar10()

## 2. Run / Load Sweep Experiments

We define $K$-sweeps for each dataset and run trials (logistic regression on centered training data, computing accuracy and covariance effective rank). If the experiments have already been run, we load them from `results/all_results.json` to save time.

In [ ]:
results_path = '../results/all_results.json'
all_results = load_results(results_path)

if all_results is None:
    print("Running sweeps from scratch. This may take 1-2 minutes...")
    all_results = {}
    
    mnist_Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
    fashion_Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
    kuzushiji_Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
    usps_Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512]
    bc_Ks = [2, 4, 8, 16, 32, 64, 100]
    cifar_Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
    
    # MNIST
    all_results['MNIST_0v1'] = run_experiment('MNIST_0v1', X_mnist, y_mnist, 0, 1, mnist_Ks, 200, n_trials=50, pca_dims=50)
    all_results['MNIST_3v8'] = run_experiment('MNIST_3v8', X_mnist, y_mnist, 3, 8, mnist_Ks, 200, n_trials=50, pca_dims=50)
    all_results['MNIST_4v9'] = run_experiment('MNIST_4v9', X_mnist, y_mnist, 4, 9, mnist_Ks, 200, n_trials=50, pca_dims=50)
    
    # Fashion
    all_results['Fashion_0v1'] = run_experiment('Fashion_0v1', X_fashion, y_fashion, 0, 1, fashion_Ks, 200, n_trials=50, pca_dims=50)
    all_results['Fashion_2v6'] = run_experiment('Fashion_2v6', X_fashion, y_fashion, 2, 6, fashion_Ks, 200, n_trials=50, pca_dims=50)
    
    # Kuzushiji
    all_results['Kuzushiji_0v1'] = run_experiment('Kuzushiji_0v1', X_kuzushiji, y_kuzushiji, 0, 1, kuzushiji_Ks, 200, n_trials=50, pca_dims=50)
    
    # USPS
    all_results['USPS_1v2'] = run_experiment('USPS_1v2', X_usps, y_usps, 1, 2, usps_Ks, 100, n_trials=50, pca_dims=50)
    
    # Breast Cancer
    all_results['BreastCancer'] = run_experiment('BreastCancer', X_bc, y_bc, 0, 1, bc_Ks, 50, n_trials=100, pca_dims=None)
    
    # CIFAR-10
    if X_cifar is not None:
        all_results['CIFAR_0v1'] = run_experiment('CIFAR_0v1', X_cifar, y_cifar, 0, 1, cifar_Ks, 200, n_trials=50, pca_dims=50)
        all_results['CIFAR_3v5'] = run_experiment('CIFAR_3v5', X_cifar, y_cifar, 3, 5, cifar_Ks, 200, n_trials=50, pca_dims=50)
        
    # Cache results
    save_results(all_results, results_path)
else:
    print("Loaded pre-computed results from:", results_path)

## 3. Visualize Sweeps

We generate the main grid layout plot mapping Accuracy and Effective Rank against training sample sizes ($K$) for all datasets.

In [ ]:
%matplotlib inline
plot_all_sweeps_grid(all_results, '../figures/all_sweeps.png')

# Render the generated figure inside the notebook
from IPython.display import Image
Image(filename='../figures/all_sweeps.png')